# CV por planta — protocolo C (RGB/GLCM + HSV)

Evaluamos el protocolo C (23 features: RGB/GLCM masked sin VARI + histogramas HSV) con GroupKFold por planta (`link_group`). Comparamos logística, Random Forest, SVM e HistGradientBoosting por `f1_macro` e IC 95%. El holdout histórico no elige el ganador; solo sirve de contexto. Prefijo de resultados: `rocole_d05_`.



Librerías del cuaderno en una sola celda: tablas, CV por grupos, modelos y el t de Student para el IC.



In [1]:
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import t
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight


Semilla fija del proyecto: barajar plantas y reproducir el protocolo CV.



In [2]:
# Semilla fija del proyecto
seed = 42

seed


Rutas relativas a las tablas featured enmascaradas y a la carpeta de resultados D-cv-errores.



In [3]:
rgb_glcm_path = Path("../../../data/featured/rocole/rgb_glcm_features_masked.csv")
hsv_hist_path = Path("../../../data/featured/rocole/hsv_hist_features_masked.csv")
results_dir = Path("../../../experiments/results/rocole/D-cv-errores")

print("rgb_glcm_path:", rgb_glcm_path.resolve(), "existe:", rgb_glcm_path.exists())
print("hsv_hist_path:", hsv_hist_path.resolve(), "existe:", hsv_hist_path.exists())
print("results_dir:", results_dir.resolve())
rgb_glcm_path, hsv_hist_path, results_dir


rgb_glcm_path: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\data\featured\rocole\rgb_glcm_features_masked.csv existe: True
hsv_hist_path: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\data\featured\rocole\hsv_hist_features_masked.csv existe: True
results_dir: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results\rocole\D-cv-errores


Creamos la carpeta de resultados D si aún no existe.



In [4]:
results_dir.mkdir(parents=True, exist_ok=True)

results_dir.exists(), results_dir


Cargamos la tabla RGB/GLCM masked ya calculada (incluye VARI; más adelante no la usamos).



In [5]:
rgb_glcm_features = pd.read_csv(rgb_glcm_path)

print("filas rgb/glcm:", len(rgb_glcm_features))
rgb_glcm_features.head()


filas rgb/glcm: 1481


Cargamos la tabla de histogramas HSV masked.



In [6]:
hsv_hist_features = pd.read_csv(hsv_hist_path)

print("filas hsv:", len(hsv_hist_features))
hsv_hist_features.head()


filas hsv: 1481


Unimos RGB/GLCM y HSV por `File`. Conservamos metadatos del RGB y el error de ambas mesas; aún no filtramos.



In [7]:
hsv_feature_names = [c for c in hsv_hist_features.columns if c.startswith("hsv_")]
hsv_to_merge = hsv_hist_features[["File", "error"] + hsv_feature_names].rename(
    columns={"error": "error_hsv"}
)

merged_leaves = rgb_glcm_features.merge(hsv_to_merge, on="File", how="inner")

print("filas merge:", len(merged_leaves))
print("columnas:", list(merged_leaves.columns))
merged_leaves.head()


filas merge: 1481
columnas: ['File', 'split', 'link_group', 'Binary.Label', 'Multiclass.Label', 'error', 'exg_mean', 'exg_std', 'ngrdi_mean', 'ngrdi_std', 'vari_mean', 'vari_std', 'glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'error_hsv', 'hsv_h_b0', 'hsv_h_b1', 'hsv_h_b2', 'hsv_h_b3', 'hsv_h_b4', 'hsv_h_b5', 'hsv_h_b6', 'hsv_h_b7', 'hsv_s_b0', 'hsv_s_b1', 'hsv_s_b2', 'hsv_s_b3', 'hsv_v_b0', 'hsv_v_b1', 'hsv_v_b2', 'hsv_v_b3']


Nos quedamos solo con hojas sin error de extracción en RGB/GLCM ni en HSV. Todas entran a la CV; no filtramos por `split`.



In [8]:
rgb_error_text = merged_leaves["error"].astype("string").fillna("").str.strip()
hsv_error_text = merged_leaves["error_hsv"].astype("string").fillna("").str.strip()
valid_leaves = merged_leaves.loc[(rgb_error_text == "") & (hsv_error_text == "")].copy()

print("hojas válidas:", len(valid_leaves))
valid_leaves.head()


hojas válidas: 1481


Inventario de hojas válidas por `Binary.Label`.



In [9]:
label_counts = valid_leaves["Binary.Label"].value_counts()

label_counts


Conteo por `split` histórico solo como contexto: no entra en la elección del modelo ni en el tuning.



In [10]:
historical_split_counts = valid_leaves["split"].value_counts()

historical_split_counts


Lista fija de las 23 columnas del protocolo C: índices RGB/GLCM sin VARI más bins HSV.



In [11]:
feature_columns = [
    "exg_mean",
    "exg_std",
    "ngrdi_mean",
    "ngrdi_std",
    "glcm_contrast",
    "glcm_homogeneity",
    "glcm_energy",
    "hsv_h_b0",
    "hsv_h_b1",
    "hsv_h_b2",
    "hsv_h_b3",
    "hsv_h_b4",
    "hsv_h_b5",
    "hsv_h_b6",
    "hsv_h_b7",
    "hsv_s_b0",
    "hsv_s_b1",
    "hsv_s_b2",
    "hsv_s_b3",
    "hsv_v_b0",
    "hsv_v_b1",
    "hsv_v_b2",
    "hsv_v_b3",
]

print("n features:", len(feature_columns))
feature_columns


n features: 23


Comprobamos que las 23 columnas del protocolo C existen en las hojas válidas.



In [12]:
missing_feature_columns = [c for c in feature_columns if c not in valid_leaves.columns]

missing_feature_columns


Armamos la tabla de CV: todas las hojas válidas con features, etiqueta binaria, planta y metadatos para OOF. Sin filtrar por `split`.



In [13]:
meta_columns = ["File", "split", "link_group", "Binary.Label", "Multiclass.Label"]
cv_leaves = valid_leaves[meta_columns + feature_columns].copy().reset_index(drop=True)

print("filas cv:", len(cv_leaves))
cv_leaves.head()


filas cv: 1481


Matriz de medidas para la CV externa, alineada fila a fila con `cv_leaves`.



In [14]:
feature_matrix = cv_leaves[feature_columns].to_numpy(dtype=float)

feature_matrix.shape


Etiqueta de entrenamiento: `Binary.Label`, alineada con la matriz. No usamos `Multiclass.Label` como target.



In [15]:
leaf_labels = cv_leaves["Binary.Label"].to_numpy()

pd.Series(leaf_labels).value_counts()


Grupos de planta (`link_group`) para que ninguna planta quede a la vez en train y en val del fold.



In [16]:
leaf_groups = cv_leaves["link_group"].to_numpy()

leaf_groups[:10], len(leaf_groups)


Cuántas plantas distintas hay y cuántas hojas por planta (resumen corto).



In [17]:
group_counts = pd.Series(leaf_groups).value_counts().sort_index()

print("n plantas:", group_counts.size)
print("hojas por planta: min", group_counts.min(), "max", group_counts.max(), "mediana", group_counts.median())
group_counts.head()


n plantas: 357
hojas por planta: min 2 max 17 mediana 4.0


Barajamos las plantas con `RandomState(seed)` y armamos 5 folds externos GroupKFold por `link_group`.



In [18]:
# Barajar plantas y reordenar filas para que GroupKFold vea grupos en ese orden.
unique_groups = np.unique(leaf_groups)
rng = np.random.RandomState(seed)
shuffled_groups = rng.permutation(unique_groups)
group_rank = {g: rank for rank, g in enumerate(shuffled_groups)}
sample_order = np.argsort(np.array([group_rank[g] for g in leaf_groups], dtype=int))

feature_matrix_ordered = feature_matrix[sample_order]
leaf_labels_ordered = leaf_labels[sample_order]
leaf_groups_ordered = leaf_groups[sample_order]

outer_cv = GroupKFold(n_splits=5)
outer_fold_splits = []
for fold_id, (train_rel, test_rel) in enumerate(
    outer_cv.split(feature_matrix_ordered, leaf_labels_ordered, leaf_groups_ordered)
):
    train_idx = sample_order[train_rel]
    test_idx = sample_order[test_rel]
    outer_fold_splits.append((fold_id, train_idx, test_idx))

for fold_id, train_idx, test_idx in outer_fold_splits:
    train_groups = set(leaf_groups[train_idx])
    test_groups = set(leaf_groups[test_idx])
    print(
        f"fold {fold_id}: train={len(train_idx)} test={len(test_idx)} "
        f"plantas_test={len(test_groups)} overlap={len(train_groups & test_groups)}"
    )

len(outer_fold_splits)


fold 0: train=1184 test=297 plantas_test=71 overlap=0
fold 1: train=1185 test=296 plantas_test=71 overlap=0
fold 2: train=1184 test=297 plantas_test=72 overlap=0
fold 3: train=1186 test=295 plantas_test=71 overlap=0
fold 4: train=1185 test=296 plantas_test=72 overlap=0


Rejilla de hiperparámetros de logística: C en un rango amplio.



In [19]:
logistic_param_grid = {
    "C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
}

logistic_param_grid


Rejilla de Random Forest: `n_estimators` y `max_depth`.



In [20]:
forest_param_grid = {
    "n_estimators": [100, 200, 400],
    "max_depth": [None, 5, 10, 20],
}

forest_param_grid


Rejilla de SVM: C y kernels `linear` y `rbf`.



In [21]:
svc_param_grid = {
    "C": [0.1, 1.0, 10.0],
    "kernel": ["linear", "rbf"],
}

svc_param_grid


Rejilla de HistGradientBoosting: `learning_rate`, `max_depth` y `max_iter`.



In [22]:
hist_param_grid = {
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [3, 5, None],
    "max_iter": [100, 200],
}

hist_param_grid


CV externa de logística: en cada fold, `StandardScaler` y búsqueda de C solo en el train del fold (GroupKFold interno); medimos `f1_macro` en el fold externo y guardamos predicciones OOF.



In [23]:
logistic_fold_scores = []
logistic_fold_params = []
logistic_oof_predictions = np.empty(len(cv_leaves), dtype=object)

for fold_id, train_idx, test_idx in outer_fold_splits:
    train_features = feature_matrix[train_idx]
    heldout_features = feature_matrix[test_idx]
    train_labels = leaf_labels[train_idx]
    heldout_labels = leaf_labels[test_idx]
    train_groups = leaf_groups[train_idx]

    n_train_groups = len(np.unique(train_groups))
    n_inner = 3 if n_train_groups >= 3 else 2
    inner_cv = GroupKFold(n_splits=n_inner)

    best_inner_score = -np.inf
    best_params = None
    for C in logistic_param_grid["C"]:
        inner_scores = []
        for inner_train_idx, inner_val_idx in inner_cv.split(
            train_features, train_labels, train_groups
        ):
            scaler_inner = StandardScaler()
            inner_train_features = scaler_inner.fit_transform(
                train_features[inner_train_idx]
            )
            inner_val_features = scaler_inner.transform(train_features[inner_val_idx])
            model_inner = LogisticRegression(
                C=C,
                class_weight="balanced",
                max_iter=2000,
                random_state=seed,
            )
            model_inner.fit(inner_train_features, train_labels[inner_train_idx])
            inner_val_predictions = model_inner.predict(inner_val_features)
            inner_scores.append(
                f1_score(
                    train_labels[inner_val_idx],
                    inner_val_predictions,
                    average="macro",
                )
            )
        mean_inner = float(np.mean(inner_scores))
        if mean_inner > best_inner_score:
            best_inner_score = mean_inner
            best_params = {"C": C}

    scaler = StandardScaler()
    train_features_scaled = scaler.fit_transform(train_features)
    heldout_features_scaled = scaler.transform(heldout_features)
    model = LogisticRegression(
        C=best_params["C"],
        class_weight="balanced",
        max_iter=2000,
        random_state=seed,
    )
    model.fit(train_features_scaled, train_labels)
    heldout_predictions = model.predict(heldout_features_scaled)
    fold_score = f1_score(heldout_labels, heldout_predictions, average="macro")

    logistic_fold_scores.append(fold_score)
    logistic_fold_params.append({"fold": fold_id, **best_params})
    logistic_oof_predictions[test_idx] = heldout_predictions
    print(f"fold {fold_id}: f1_macro={fold_score:.4f} params={best_params}")

logistic_fold_scores, logistic_fold_params


fold 0: f1_macro=0.7506 params={'C': 100.0}
fold 1: f1_macro=0.7564 params={'C': 1.0}
fold 2: f1_macro=0.7270 params={'C': 10.0}
fold 3: f1_macro=0.7435 params={'C': 0.1}
fold 4: f1_macro=0.7297 params={'C': 1.0}


Tabla de `f1_macro` por fold externo de logística.



In [24]:
logistic_scores_table = pd.DataFrame(
    {
        "fold": [fold_id for fold_id, _, _ in outer_fold_splits],
        "f1_macro": logistic_fold_scores,
    }
)

logistic_scores_table


Media e IC 95% (t de Student, 4 g.l.) del `f1_macro` de logística sobre los cinco folds.



In [25]:
n_folds = len(logistic_fold_scores)
logistic_mean = float(np.mean(logistic_fold_scores))
logistic_std = float(np.std(logistic_fold_scores, ddof=1))
logistic_ci_half = float(t.ppf(0.975, df=n_folds - 1) * logistic_std / np.sqrt(n_folds))
logistic_cv_summary = {
    "model_name": "logistic",
    "mean_f1_macro": logistic_mean,
    "ci95_low": logistic_mean - logistic_ci_half,
    "ci95_high": logistic_mean + logistic_ci_half,
    "std_f1_macro": logistic_std,
}

logistic_cv_summary


CV externa de Random Forest: sin scaler, `class_weight=balanced`, búsqueda interna de hiperparámetros y predicciones OOF.



In [26]:
forest_fold_scores = []
forest_fold_params = []
forest_oof_predictions = np.empty(len(cv_leaves), dtype=object)

forest_param_list = list(
    product(forest_param_grid["n_estimators"], forest_param_grid["max_depth"])
)

for fold_id, train_idx, test_idx in outer_fold_splits:
    train_features = feature_matrix[train_idx]
    heldout_features = feature_matrix[test_idx]
    train_labels = leaf_labels[train_idx]
    heldout_labels = leaf_labels[test_idx]
    train_groups = leaf_groups[train_idx]

    n_train_groups = len(np.unique(train_groups))
    n_inner = 3 if n_train_groups >= 3 else 2
    inner_cv = GroupKFold(n_splits=n_inner)

    best_inner_score = -np.inf
    best_params = None
    for n_estimators, max_depth in forest_param_list:
        inner_scores = []
        for inner_train_idx, inner_val_idx in inner_cv.split(
            train_features, train_labels, train_groups
        ):
            model_inner = RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                class_weight="balanced",
                random_state=seed,
                n_jobs=-1,
            )
            model_inner.fit(
                train_features[inner_train_idx],
                train_labels[inner_train_idx],
            )
            inner_val_predictions = model_inner.predict(train_features[inner_val_idx])
            inner_scores.append(
                f1_score(
                    train_labels[inner_val_idx],
                    inner_val_predictions,
                    average="macro",
                )
            )
        mean_inner = float(np.mean(inner_scores))
        if mean_inner > best_inner_score:
            best_inner_score = mean_inner
            best_params = {"n_estimators": n_estimators, "max_depth": max_depth}

    model = RandomForestClassifier(
        n_estimators=best_params["n_estimators"],
        max_depth=best_params["max_depth"],
        class_weight="balanced",
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(train_features, train_labels)
    heldout_predictions = model.predict(heldout_features)
    fold_score = f1_score(heldout_labels, heldout_predictions, average="macro")

    forest_fold_scores.append(fold_score)
    forest_fold_params.append({"fold": fold_id, **best_params})
    forest_oof_predictions[test_idx] = heldout_predictions
    print(f"fold {fold_id}: f1_macro={fold_score:.4f} params={best_params}")

forest_fold_scores, forest_fold_params


fold 0: f1_macro=0.7769 params={'n_estimators': 400, 'max_depth': None}
fold 1: f1_macro=0.7868 params={'n_estimators': 200, 'max_depth': 10}
fold 2: f1_macro=0.7946 params={'n_estimators': 100, 'max_depth': None}
fold 3: f1_macro=0.8098 params={'n_estimators': 400, 'max_depth': None}
fold 4: f1_macro=0.8106 params={'n_estimators': 100, 'max_depth': 10}


Tabla de `f1_macro` por fold externo de Random Forest.



In [27]:
forest_scores_table = pd.DataFrame(
    {
        "fold": [fold_id for fold_id, _, _ in outer_fold_splits],
        "f1_macro": forest_fold_scores,
    }
)

forest_scores_table


Media e IC 95% del `f1_macro` de Random Forest.



In [28]:
n_folds = len(forest_fold_scores)
forest_mean = float(np.mean(forest_fold_scores))
forest_std = float(np.std(forest_fold_scores, ddof=1))
forest_ci_half = float(t.ppf(0.975, df=n_folds - 1) * forest_std / np.sqrt(n_folds))
forest_cv_summary = {
    "model_name": "random_forest",
    "mean_f1_macro": forest_mean,
    "ci95_low": forest_mean - forest_ci_half,
    "ci95_high": forest_mean + forest_ci_half,
    "std_f1_macro": forest_std,
}

forest_cv_summary


CV externa de SVM: scaler solo en el train del fold, `class_weight=balanced`, búsqueda interna y OOF.



In [29]:
svc_fold_scores = []
svc_fold_params = []
svc_oof_predictions = np.empty(len(cv_leaves), dtype=object)

svc_param_list = list(product(svc_param_grid["C"], svc_param_grid["kernel"]))

for fold_id, train_idx, test_idx in outer_fold_splits:
    train_features = feature_matrix[train_idx]
    heldout_features = feature_matrix[test_idx]
    train_labels = leaf_labels[train_idx]
    heldout_labels = leaf_labels[test_idx]
    train_groups = leaf_groups[train_idx]

    n_train_groups = len(np.unique(train_groups))
    n_inner = 3 if n_train_groups >= 3 else 2
    inner_cv = GroupKFold(n_splits=n_inner)

    best_inner_score = -np.inf
    best_params = None
    for C, kernel in svc_param_list:
        inner_scores = []
        for inner_train_idx, inner_val_idx in inner_cv.split(
            train_features, train_labels, train_groups
        ):
            scaler_inner = StandardScaler()
            inner_train_features = scaler_inner.fit_transform(
                train_features[inner_train_idx]
            )
            inner_val_features = scaler_inner.transform(train_features[inner_val_idx])
            model_inner = SVC(
                C=C,
                kernel=kernel,
                class_weight="balanced",
                probability=False,
                cache_size=500,
                random_state=seed,
            )
            model_inner.fit(inner_train_features, train_labels[inner_train_idx])
            inner_val_predictions = model_inner.predict(inner_val_features)
            inner_scores.append(
                f1_score(
                    train_labels[inner_val_idx],
                    inner_val_predictions,
                    average="macro",
                )
            )
        mean_inner = float(np.mean(inner_scores))
        if mean_inner > best_inner_score:
            best_inner_score = mean_inner
            best_params = {"C": C, "kernel": kernel}

    scaler = StandardScaler()
    train_features_scaled = scaler.fit_transform(train_features)
    heldout_features_scaled = scaler.transform(heldout_features)
    model = SVC(
        C=best_params["C"],
        kernel=best_params["kernel"],
        class_weight="balanced",
        probability=False,
        cache_size=500,
        random_state=seed,
    )
    model.fit(train_features_scaled, train_labels)
    heldout_predictions = model.predict(heldout_features_scaled)
    fold_score = f1_score(heldout_labels, heldout_predictions, average="macro")

    svc_fold_scores.append(fold_score)
    svc_fold_params.append({"fold": fold_id, **best_params})
    svc_oof_predictions[test_idx] = heldout_predictions
    print(f"fold {fold_id}: f1_macro={fold_score:.4f} params={best_params}")

svc_fold_scores, svc_fold_params


fold 0: f1_macro=0.7104 params={'C': 10.0, 'kernel': 'rbf'}
fold 1: f1_macro=0.8000 params={'C': 10.0, 'kernel': 'rbf'}
fold 2: f1_macro=0.7606 params={'C': 10.0, 'kernel': 'rbf'}
fold 3: f1_macro=0.8078 params={'C': 10.0, 'kernel': 'rbf'}
fold 4: f1_macro=0.7567 params={'C': 10.0, 'kernel': 'rbf'}


Tabla de `f1_macro` por fold externo de SVM.



In [30]:
svc_scores_table = pd.DataFrame(
    {
        "fold": [fold_id for fold_id, _, _ in outer_fold_splits],
        "f1_macro": svc_fold_scores,
    }
)

svc_scores_table


Media e IC 95% del `f1_macro` de SVM.



In [31]:
n_folds = len(svc_fold_scores)
svc_mean = float(np.mean(svc_fold_scores))
svc_std = float(np.std(svc_fold_scores, ddof=1))
svc_ci_half = float(t.ppf(0.975, df=n_folds - 1) * svc_std / np.sqrt(n_folds))
svc_cv_summary = {
    "model_name": "svc",
    "mean_f1_macro": svc_mean,
    "ci95_low": svc_mean - svc_ci_half,
    "ci95_high": svc_mean + svc_ci_half,
    "std_f1_macro": svc_std,
}

svc_cv_summary


CV externa de HistGradientBoosting: sin scaler; balanceo con `compute_sample_weight('balanced')` en el fit; búsqueda interna y OOF.



In [32]:
hist_fold_scores = []
hist_fold_params = []
hist_oof_predictions = np.empty(len(cv_leaves), dtype=object)

hist_param_list = list(
    product(
        hist_param_grid["learning_rate"],
        hist_param_grid["max_depth"],
        hist_param_grid["max_iter"],
    )
)

for fold_id, train_idx, test_idx in outer_fold_splits:
    train_features = feature_matrix[train_idx]
    heldout_features = feature_matrix[test_idx]
    train_labels = leaf_labels[train_idx]
    heldout_labels = leaf_labels[test_idx]
    train_groups = leaf_groups[train_idx]

    n_train_groups = len(np.unique(train_groups))
    n_inner = 3 if n_train_groups >= 3 else 2
    inner_cv = GroupKFold(n_splits=n_inner)

    best_inner_score = -np.inf
    best_params = None
    for learning_rate, max_depth, max_iter in hist_param_list:
        inner_scores = []
        for inner_train_idx, inner_val_idx in inner_cv.split(
            train_features, train_labels, train_groups
        ):
            sample_weight_inner = compute_sample_weight(
                "balanced", train_labels[inner_train_idx]
            )
            model_inner = HistGradientBoostingClassifier(
                learning_rate=learning_rate,
                max_depth=max_depth,
                max_iter=max_iter,
                random_state=seed,
            )
            model_inner.fit(
                train_features[inner_train_idx],
                train_labels[inner_train_idx],
                sample_weight=sample_weight_inner,
            )
            inner_val_predictions = model_inner.predict(train_features[inner_val_idx])
            inner_scores.append(
                f1_score(
                    train_labels[inner_val_idx],
                    inner_val_predictions,
                    average="macro",
                )
            )
        mean_inner = float(np.mean(inner_scores))
        if mean_inner > best_inner_score:
            best_inner_score = mean_inner
            best_params = {
                "learning_rate": learning_rate,
                "max_depth": max_depth,
                "max_iter": max_iter,
            }

    sample_weight = compute_sample_weight("balanced", train_labels)
    model = HistGradientBoostingClassifier(
        learning_rate=best_params["learning_rate"],
        max_depth=best_params["max_depth"],
        max_iter=best_params["max_iter"],
        random_state=seed,
    )
    model.fit(train_features, train_labels, sample_weight=sample_weight)
    heldout_predictions = model.predict(heldout_features)
    fold_score = f1_score(heldout_labels, heldout_predictions, average="macro")

    hist_fold_scores.append(fold_score)
    hist_fold_params.append({"fold": fold_id, **best_params})
    hist_oof_predictions[test_idx] = heldout_predictions
    print(f"fold {fold_id}: f1_macro={fold_score:.4f} params={best_params}")

hist_fold_scores, hist_fold_params


fold 0: f1_macro=0.7398 params={'learning_rate': 0.1, 'max_depth': None, 'max_iter': 100}
fold 1: f1_macro=0.7694 params={'learning_rate': 0.2, 'max_depth': None, 'max_iter': 200}
fold 2: f1_macro=0.7979 params={'learning_rate': 0.2, 'max_depth': None, 'max_iter': 200}
fold 3: f1_macro=0.8127 params={'learning_rate': 0.05, 'max_depth': None, 'max_iter': 100}
fold 4: f1_macro=0.7801 params={'learning_rate': 0.2, 'max_depth': None, 'max_iter': 100}


Tabla de `f1_macro` por fold externo de HistGradientBoosting.



In [33]:
hist_scores_table = pd.DataFrame(
    {
        "fold": [fold_id for fold_id, _, _ in outer_fold_splits],
        "f1_macro": hist_fold_scores,
    }
)

hist_scores_table


Media e IC 95% del `f1_macro` de HistGradientBoosting.



In [34]:
n_folds = len(hist_fold_scores)
hist_mean = float(np.mean(hist_fold_scores))
hist_std = float(np.std(hist_fold_scores, ddof=1))
hist_ci_half = float(t.ppf(0.975, df=n_folds - 1) * hist_std / np.sqrt(n_folds))
hist_cv_summary = {
    "model_name": "hist_gradient_boosting",
    "mean_f1_macro": hist_mean,
    "ci95_low": hist_mean - hist_ci_half,
    "ci95_high": hist_mean + hist_ci_half,
    "std_f1_macro": hist_std,
}

hist_cv_summary


Tabla comparativa de los cuatro modelos: media CV, IC 95% y `f1_macro` por fold. Sin mirar el test histórico.



In [35]:
model_comparison_table = pd.DataFrame(
    [
        {
            **logistic_cv_summary,
            **{f"fold_{i}_f1_macro": score for i, score in enumerate(logistic_fold_scores)},
        },
        {
            **forest_cv_summary,
            **{f"fold_{i}_f1_macro": score for i, score in enumerate(forest_fold_scores)},
        },
        {
            **svc_cv_summary,
            **{f"fold_{i}_f1_macro": score for i, score in enumerate(svc_fold_scores)},
        },
        {
            **hist_cv_summary,
            **{f"fold_{i}_f1_macro": score for i, score in enumerate(hist_fold_scores)},
        },
    ]
)

model_comparison_table


Elegimos el ganador solo por la media de `f1_macro` en la CV externa. El split histórico de test no desempata.



In [36]:
best_row = model_comparison_table.loc[
    model_comparison_table["mean_f1_macro"].idxmax()
]
best_model_name = best_row["model_name"]

print("best_model_name:", best_model_name)
best_model_name


best_model_name: random_forest


Guardamos el CSV de métricas por fold e IC con prefijo `rocole_d05_`.



In [37]:
cv_metrics_path = results_dir / "rocole_d05_cv_metrics.csv"
model_comparison_table.to_csv(cv_metrics_path, index=False)

print("guardado:", cv_metrics_path.resolve())
cv_metrics_path


guardado: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results\rocole\D-cv-errores\rocole_d05_cv_metrics.csv


Consolidamos las predicciones OOF de los cuatro modelos y la tabla del ganador (fold por hoja), listas para guardar.


In [38]:
oof_by_model = {
    "logistic": logistic_oof_predictions,
    "random_forest": forest_oof_predictions,
    "svc": svc_oof_predictions,
    "hist_gradient_boosting": hist_oof_predictions,
}

fold_of_leaf = np.empty(len(cv_leaves), dtype=int)
for fold_id, _, test_idx in outer_fold_splits:
    fold_of_leaf[test_idx] = fold_id

oof_predictions = pd.DataFrame(
    {
        "File": cv_leaves["File"].to_numpy(),
        "link_group": cv_leaves["link_group"].to_numpy(),
        "Binary.Label": cv_leaves["Binary.Label"].to_numpy(),
        "Multiclass.Label": cv_leaves["Multiclass.Label"].to_numpy(),
        "fold": fold_of_leaf,
        "pred_logistic": logistic_oof_predictions,
        "pred_random_forest": forest_oof_predictions,
        "pred_svc": svc_oof_predictions,
        "pred_hist_gradient_boosting": hist_oof_predictions,
    }
)

best_oof_predictions = pd.DataFrame(
    {
        "File": cv_leaves["File"].to_numpy(),
        "link_group": cv_leaves["link_group"].to_numpy(),
        "Binary.Label": cv_leaves["Binary.Label"].to_numpy(),
        "Multiclass.Label": cv_leaves["Multiclass.Label"].to_numpy(),
        "fold": fold_of_leaf,
        "y_true": cv_leaves["Binary.Label"].to_numpy(),
        "y_pred": oof_by_model[best_model_name],
        "model_name": best_model_name,
    }
)

print("best_model_name:", best_model_name)
oof_predictions.head(), best_oof_predictions.head()


best_model_name: random_forest


Guardamos predicciones OOF de todos los modelos y las del ganador, para sensibilidad multiclass y falsos negativos en D-06/D-07.



In [39]:
oof_predictions_path = results_dir / "rocole_d05_oof_predictions.csv"
best_oof_path = results_dir / "rocole_d05_best_oof_predictions.csv"

oof_predictions.to_csv(oof_predictions_path, index=False)
best_oof_predictions.to_csv(best_oof_path, index=False)

print("best_model_name:", best_model_name)
print("oof:", oof_predictions_path.resolve())
print("best oof:", best_oof_path.resolve())
oof_predictions_path, best_oof_path


best_model_name: random_forest
oof: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results\rocole\D-cv-errores\rocole_d05_oof_predictions.csv
best oof: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results\rocole\D-cv-errores\rocole_d05_best_oof_predictions.csv


## Conclusiones

Ganador por media de `f1_macro` en GroupKFold por planta (protocolo C): **random_forest** 0.796 [IC 95% 0.778–0.814]. HistGB queda cerca (0.780), SVM 0.767 y logística 0.741. El número que reportamos al profesor es esta CV externa; no usamos el holdout histórico de C para elegir.
